# TP3 & TP4 : Classification Multiclasse et Régression sur Données Réelles Bruitées
## Avec Dimension VC, Optimisation Trilevel/Multilevel et Analyse de Taille d'Échantillon

### Rappels Théoriques

#### 1. Dimension VC (Vapnik-Chervonenkis)
Pour un classifieur linéaire dans un espace de dimension $d$, la dimension VC est $d+1$. Pour les machines à vecteurs de support (SVM) avec marge, la dimension VC est bornée par :
$$ VC(\mathcal{H}) \le \min\left( \left\lfloor \frac{R^2}{\gamma^2} \right\rfloor, d \right) + 1 $$
Où $R$ est le rayon de la plus petite sphère contenant les données, et $\gamma$ est la marge ($\gamma = \frac{2}{||w||}$). 
**Explication :** Plus la marge est grande (petit $||w||$), plus la dimension VC est faible, ce qui réduit le risque de surapprentissage.

#### 2. Complexité de Rademacher
Elle mesure la capacité d'une classe de fonctions à s'adapter à un bruit aléatoire. Pour les classifieurs linéaires avec une norme bornée $\Lambda$ et des données de rayon $R$ :
$$ \mathfrak{R}_n(\mathcal{H}) \le \sqrt{\frac{R^2 \Lambda^2}{n}} $$
**Explication :** Plus le nombre d'échantillons $n$ est grand, plus la complexité de Rademacher diminue, ce qui garantit une meilleure généralisation.

#### 3. Compromis Biais-Variance
L'erreur totale d'un modèle se décompose en :
$$ E[(y - \hat{f}(x))^2] = \text{Biais}^2 + \text{Variance} + \text{Bruit Irréductible} $$
- **Biais :** Erreur due à des hypothèses trop simplistes (sous-apprentissage).
- **Variance :** Sensibilité du modèle aux fluctuations des données d'entraînement (surapprentissage).

#### 4. Optimisation Trilevel et Multilevel
- **Trilevel (TP3) :** $\min_{p} \min_{C, a} \min_{w, b} \mathcal{L}(w, b, C, a, p)$
  - Niveau 1 (Extérieur) : Optimisation de $p$ (ex: type de noyau, norme $L_p$).
  - Niveau 2 (Milieu) : Optimisation des hyperparamètres $C$ (régularisation) et $a$ (ex: $\gamma$ pour RBF).
  - Niveau 3 (Intérieur) : Optimisation des paramètres du modèle $w, b$.
- **Multilevel (TP4) :** $\min_{\epsilon} \min_{C, \text{kernel}} \min_{w, b} \mathcal{L}(w, b, C, \epsilon)$
  - Optimisation séquentielle ou conjointe de la marge $\epsilon$, de $C$, et des paramètres du noyau.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.svm import SVC, SVR
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# --- Fonction pour ajouter du bruit ---
def add_noise(X, y, noise_level, task='classification'):
    X_noisy = X.copy()
    y_noisy = y.copy()
    if task == 'classification':
        X_noisy += np.random.normal(0, noise_level * np.std(X, axis=0), X.shape)
    else:
        y_noisy += np.random.normal(0, noise_level * np.std(y), y.shape)
    return X_noisy, y_noisy

# --- Génération des données (TP3 : Multiclasse) ---
X_clf, y_clf = datasets.make_classification(
    n_samples=500, n_features=2, n_informative=2, n_redundant=0, 
    n_classes=3, n_clusters_per_class=1, random_state=42
)

# --- Génération des données (TP4 : Régression) ---
X_reg, y_reg = datasets.make_regression(
    n_samples=500, n_features=1, noise=0.0, random_state=42
)

# --- Visualisation des données bruitées (10%, 20%, 30%) ---
noise_levels = [0.1, 0.2, 0.3]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, nl in enumerate(noise_levels):
    X_n, y_n = add_noise(X_clf, y_clf, nl, 'classification')
    axes[i].scatter(X_n[:, 0], X_n[:, 1], c=y_n, cmap='viridis', edgecolor='k', alpha=0.7)
    axes[i].set_title(f"Classification - Bruit {int(nl*100)}%")
    axes[i].set_xlabel("Feature 1")
    axes[i].set_ylabel("Feature 2")

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, nl in enumerate(noise_levels):
    X_n, y_n = add_noise(X_reg, y_reg, nl, 'regression')
    axes[i].scatter(X_n, y_n, color='blue', alpha=0.5, s=10)
    axes[i].set_title(f"Régression - Bruit {int(nl*100)}%")
    axes[i].set_xlabel("X")
    axes[i].set_ylabel("y")

plt.tight_layout()
plt.show()

### TP3 : Classification Multiclasse avec SVC
Nous utilisons un SVC probabiliste (One-vs-One). 
Ensuite, nous calculons la dimension VC empirique en fonction de la marge.

In [ ]:
# --- Préparation des données (avec 20% de bruit pour l'exemple) ---
X_train, X_test, y_train, y_test = train_test_split(X_clf, y_clf, test_size=0.3, random_state=42)
X_train_n, y_train_n = add_noise(X_train, y_train, 0.2, 'classification')
X_test_n, y_test_n = add_noise(X_test, y_test, 0.2, 'classification')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_n)
X_test_scaled = scaler.transform(X_test_n)

# --- Entraînement du SVC Multiclasse ---
svc = SVC(kernel='linear', C=1.0, probability=True, decision_function_shape='ovo', random_state=42)
svc.fit(X_train_scaled, y_train_n)
y_pred = svc.predict(X_test_scaled)

print(f"Précision du SVC Multiclasse (Bruit 20%) : {accuracy_score(y_test_n, y_pred):.4f}")

# --- Calcul de la Dimension VC Empirique ---
w = svc.coef_
w_norm = np.linalg.norm(w, axis=1)
gamma = 2 / w_norm

center = np.mean(X_train_scaled, axis=0)
R = np.max(np.linalg.norm(X_train_scaled - center, axis=1))
d = X_train_scaled.shape[1]

vc_bounds = []
for g in gamma:
    vc_bound = min(np.floor(R**2 / g**2), d) + 1
    vc_bounds.append(vc_bound)

print(f"Rayon R : {R:.4f}")
print(f"Marges gamma : {gamma}")
print(f"Bornes VC estimées : {vc_bounds}")
print(f"Dimension de l'espace : {d}")

C_values = np.logspace(-3, 3, 10)
vc_dims = []
for C in C_values:
    svc_temp = SVC(kernel='linear', C=C, random_state=42).fit(X_train_scaled, y_train_n)
    w_temp = svc_temp.coef_
    gamma_temp = 2 / np.linalg.norm(w_temp, axis=1)
    vc_temp = min(np.floor(R**2 / np.mean(gamma_temp)**2), d) + 1
    vc_dims.append(vc_temp)

plt.figure(figsize=(8, 5))
plt.plot(C_values, vc_dims, marker='o', linestyle='-', color='red')
plt.xscale('log')
plt.xlabel("Paramètre de régularisation C (log)")
plt.ylabel("Dimension VC estimée")
plt.title("Évolution de la Dimension VC en fonction de C")
plt.grid(True)
plt.show()

### Optimisation Trilevel pour le SVC
Nous allons optimiser :
- Niveau 1 (p) : Le type de noyau ('linear', 'rbf').
- Niveau 2 (C, a) : Le paramètre de régularisation C et le paramètre de noyau a (gamma).
- Niveau 3 : Les paramètres internes du modèle (w, b) sont optimisés par la fonction `fit`.

In [ ]:
from sklearn.model_selection import ParameterGrid

niveau1_p = ['linear', 'rbf']
niveau2_params = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 0.1, 0.01]
}

best_score = 0
best_params = {}

print("Début de l'optimisation Trilevel...")
for p in niveau1_p:
    for params in ParameterGrid(niveau2_params):
        if p == 'linear':
            model = SVC(kernel=p, C=params['C'], random_state=42)
        else:
            model = SVC(kernel=p, C=params['C'], gamma=params['gamma'], random_state=42)
        
        model.fit(X_train_scaled, y_train_n)
        score = model.score(X_test_scaled, y_test_n)
        
        if score > best_score:
            best_score = score
            best_params = {'p': p, 'C': params['C'], 'a(gamma)': params.get('gamma', 'N/A')}

print(f"Meilleurs paramètres Trilevel : {best_params}")
print(f"Meilleure précision : {best_score:.4f}")

### TP4 : Régression avec SVR (Support Vector Regression)
Nous appliquons un SVR à contraintes probabilistes (modèle $\epsilon$-band, cas linéaire).
Ensuite, nous effectuons une optimisation **Multilevel** pour déterminer $\epsilon$, $C$ et les paramètres du noyau.

In [ ]:
# --- Préparation des données de Régression ---
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)
X_train_rn, y_train_rn = add_noise(X_train_r, y_train_r, 0.2, 'regression')
X_test_rn, y_test_rn = add_noise(X_test_r, y_test_r, 0.2, 'regression')

scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_rn)
X_test_r_scaled = scaler_r.transform(X_test_rn)

# --- SVR Linéaire de base ---
svr = SVR(kernel='linear', C=1.0, epsilon=0.1)
svr.fit(X_train_r_scaled, y_train_rn)
y_pred_svr = svr.predict(X_test_r_scaled)
mse_base = mean_squared_error(y_test_rn, y_pred_svr)
print(f"MSE du SVR Linéaire de base : {mse_base:.4f}")

# --- Visualisation de la régression et de la bande epsilon ---
plt.figure(figsize=(10, 6))
plt.scatter(X_train_r_scaled, y_train_rn, color='gray', alpha=0.5, label='Données bruitées (20%)')
plt.plot(X_train_r_scaled, svr.predict(X_train_r_scaled), color='red', label='SVR Linéaire')
plt.fill_between(X_train_r_scaled.ravel(), 
                 svr.predict(X_train_r_scaled) - svr.epsilon, 
                 svr.predict(X_train_r_scaled) + svr.epsilon, 
                 color='red', alpha=0.2, label='Bande epsilon')
plt.xlabel("X (standardisé)")
plt.ylabel("y")
plt.title("SVR Linéaire avec Bande Epsilon")
plt.legend()
plt.show()

# --- Optimisation Multilevel pour SVR ---
epsilons = [0.01, 0.05, 0.1, 0.2, 0.5]
Cs = [0.1, 1, 10, 100]
results = np.zeros((len(epsilons), len(Cs)))

for i, eps in enumerate(epsilons):
    for j, C in enumerate(Cs):
        svr_temp = SVR(kernel='linear', C=C, epsilon=eps)
        svr_temp.fit(X_train_r_scaled, y_train_rn)
        y_pred_temp = svr_temp.predict(X_test_r_scaled)
        results[i, j] = mean_squared_error(y_test_rn, y_pred_temp)

plt.figure(figsize=(8, 6))
plt.imshow(results, cmap='viridis', aspect='auto', origin='lower')
plt.colorbar(label='MSE')
plt.xticks(np.arange(len(Cs)), Cs)
plt.yticks(np.arange(len(epsilons)), epsilons)
plt.xlabel("C")
plt.ylabel("Epsilon")
plt.title("Optimisation Multilevel : MSE en fonction de C et Epsilon")
plt.show()

best_idx = np.unravel_index(np.argmin(results, axis=None), results.shape)
print(f"Meilleurs paramètres Multilevel : C={Cs[best_idx[1]]}, Epsilon={epsilons[best_idx[0]]}")
print(f"MSE minimal : {results[best_idx]:.4f}")

### Analyse de la Taille Minimale d'Échantillon
Nous allons tracer les courbes d'apprentissage pour différentes tailles d'échantillons et différents niveaux de bruit (10%, 20%, 30%).
Cela permet de déterminer à partir de combien d'échantillons le modèle se stabilise, et comment le bruit affecte cette taille minimale.

In [ ]:
def plot_learning_curve_manual(estimator, X, y, noise_level, task='classification'):
    train_sizes = np.linspace(0.1, 1.0, 10)
    train_scores = []
    test_scores = []
    
    for size in train_sizes:
        n_samples = int(size * len(X))
        X_sub, y_sub = X[:n_samples], y[:n_samples]
        X_tr, X_te, y_tr, y_te = train_test_split(X_sub, y_sub, test_size=0.2, random_state=42)
        
        X_tr_n, y_tr_n = add_noise(X_tr, y_tr, noise_level, task)
        X_te_n, y_te_n = add_noise(X_te, y_te, noise_level, task)
        
        estimator.fit(X_tr_n, y_tr_n)
        
        if task == 'classification':
            train_scores.append(estimator.score(X_tr_n, y_tr_n))
            test_scores.append(estimator.score(X_te_n, y_te_n))
        else:
            train_scores.append(-mean_squared_error(y_tr_n, estimator.predict(X_tr_n)))
            test_scores.append(-mean_squared_error(y_te_n, estimator.predict(X_te_n)))
            
    return train_sizes * len(X), np.mean(train_scores, axis=1), np.mean(test_scores, axis=1)

plt.figure(figsize=(12, 5))
for nl in [0.1, 0.2, 0.3]:
    sizes, train_sc, test_sc = plot_learning_curve_manual(
        SVC(kernel='rbf', C=1.0, gamma='scale'), 
        X_clf, y_clf, nl, 'classification'
    )
    plt.plot(sizes, test_sc, marker='o', label=f'Test (Bruit {int(nl*100)}%)')
    
plt.xlabel("Taille de l'échantillon d'entraînement")
plt.ylabel("Précision (Accuracy)")
plt.title("Courbe d'apprentissage - Classification (SVC)")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
for nl in [0.1, 0.2, 0.3]:
    sizes, train_sc, test_sc = plot_learning_curve_manual(
        SVR(kernel='linear', C=1.0, epsilon=0.1), 
        X_reg, y_reg, nl, 'regression'
    )
    plt.plot(sizes, test_sc, marker='o', label=f'Test (Bruit {int(nl*100)}%)')
    
plt.xlabel("Taille de l'échantillon d'entraînement")
plt.ylabel("Négatif du MSE")
plt.title("Courbe d'apprentissage - Régression (SVR)")
plt.legend()
plt.grid(True)
plt.show()

### Synthèse des Résultats
1. **Dimension VC :** Nous avons observé que la dimension VC augmente avec la complexité du modèle (C élevé), ce qui augmente le risque de surapprentissage.
2. **Optimisation Trilevel :** L'optimisation conjointe du noyau, de C et de gamma a permis de trouver la meilleure configuration pour la classification multiclasse.
3. **Optimisation Multilevel :** Pour la régression, la recherche sur grille de $\epsilon$ et $C$ a montré l'importance de la bande $\epsilon$ pour contrôler la sensibilité au bruit.
4. **Taille d'échantillon et Bruit :** Les courbes d'apprentissage montrent que plus le bruit est élevé (30%), plus il faut d'échantillons pour atteindre une performance stable. À 10% de bruit, le modèle se stabilise plus rapidement.